# IoT ETL — results report

Runs the pipeline (`etl/__main__.py`) and shows what the loaded warehouse looks like.
Readings are synthetic: 3 metrics (temperature_c, vibration_mm_s, pressure_bar) every ~2h
for 30 days across 6 machines in 2 plants (Linz, Wels). Some null/missing readings and
unknown-machine rows on purpose to exercise the transform step.

In [ ]:
import sys, sqlite3
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path('..').resolve()))
from etl import run

run()

In [ ]:
DB = Path('..') / 'out' / 'warehouse.db'
con = sqlite3.connect(DB)
machines = pd.read_sql_query('select * from dim_machine', con)
readings = pd.read_sql_query('select * from fact_reading', con, parse_dates=['ts'])
print(f'machines: {len(machines)}  readings: {len(readings):,}')
machines

## readings per machine × metric

In [ ]:
pivot = readings.pivot_table(index='machine_id', columns='metric', values='value', aggfunc='count', fill_value=0)
pivot

## metric distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
for ax, (metric, color) in zip(axes, [('temperature_c', '#4C78A8'), ('vibration_mm_s', '#F58518'), ('pressure_bar', '#54A24B')]):
    sub = readings[readings.metric == metric]['value'].dropna()
    ax.hist(sub, bins=30, color=color)
    ax.set_title(metric)
    ax.set_ylabel('readings')
plt.tight_layout()

## temperature trend per machine

In [ ]:
temp = readings[readings.metric == 'temperature_c'].copy()
temp['day'] = temp['ts'].dt.date
daily = temp.groupby(['day', 'machine_id'])['value'].mean().unstack()
fig, ax = plt.subplots(figsize=(11, 4))
daily.plot(ax=ax)
ax.set_title('daily mean temperature_c by machine')
ax.set_ylabel('°C')
ax.set_xlabel('')
ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))
ax.grid(alpha=0.3)
plt.tight_layout()

## simple anomaly flag

flag readings outside mean ± 3*std per (machine, metric).

In [ ]:
stats = readings.dropna(subset=['value']).groupby(['machine_id', 'metric'])['value'].agg(['mean', 'std']).reset_index()
merged = readings.merge(stats, on=['machine_id', 'metric'])
merged['z'] = (merged['value'] - merged['mean']) / merged['std']
anom = merged[merged['z'].abs() > 3]
print(f'anomalies: {len(anom)} of {len(readings):,} readings ({len(anom)/len(readings)*100:.2f}%)')
anom.groupby(['machine_id', 'metric']).size().to_frame('count')

## plant-level summary join

In [ ]:
summary = pd.read_sql_query("""
select m.plant, r.metric,
       count(*) as readings,
       round(avg(r.value), 2) as mean,
       round(min(r.value), 2) as min,
       round(max(r.value), 2) as max
from fact_reading r
join dim_machine m on m.machine_id = r.machine_id
where r.value is not null
group by 1, 2
order by 1, 2
""", con)
summary

---
*generated by `notebooks/results.ipynb`. pipeline: extract → transform (clean nulls + drop unknown machines) → load to sqlite.*